# Lab 2.5 &mdash; Challenge &mdash; The Architecture Bake-Off

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Write the acceptance bar down <i>before</i> you have seen a single result
- Build four architectures behind one interface: direct, chain-of-thought, ReAct, reflection
- Let one disqualifying case beat a pass-rate average

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 2 labs work one case: an internal employee help desk.
> The rules are ordinary on purpose &mdash; the only new thing here is how the agent reasons.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# An internal employee help desk. Ordinary rules on purpose: the only new thing in these five
# labs is LangChain. Nothing here is real data and nothing leaves this notebook.

REQUESTS = {
    "EHD-7001": {"who": "Priya Nair",   "category": "access",   "urgency": "high",
                 "wants": "reset",
                 "text": "Locked out of the payroll portal after the password reset."},
    "EHD-7002": {"who": "Rahul Menon",  "category": "hardware", "urgency": "high",
                 "wants": "replacement",
                 "text": "Laptop battery has swollen and the case is bulging."},
    "EHD-7003": {"who": "Anita Sharma", "category": "software", "urgency": "low",
                 "wants": "licence",
                 "text": "Need a licence for the diagramming tool, about 180 USD a year."},
    "EHD-7004": {"who": "Vikram Rao",   "category": "access",   "urgency": "medium",
                 "wants": "admin-rights",
                 "text": "Please give me admin rights on the finance reporting system."},
    "EHD-7005": {"who": "Priya Nair",   "category": "hardware", "urgency": "low",
                 "wants": "replacement",
                 "text": "Second monitor flickers every few minutes."},
}

# The handbook, one entry per category. Every judgement in this module comes from these.
HANDBOOK = {
    "access":   "Verify identity, then reset. The help desk NEVER grants elevated or admin "
                "rights -- route those to Identity and Access Management.",
    "hardware": "Replace under warranty. A swollen battery is a safety issue: stop use "
                "immediately and replace the same day, whatever urgency the employee set.",
    "software": "Licences over 100 USD per year need the cost-centre owner's approval first.",
}

SLA_HOURS = {"high": 4, "medium": 24, "low": 72}
ROUTE_OUT = {"admin-rights"}     # what the help desk must hand to another team, never do itself

print(f"{len(REQUESTS)} help desk requests, {len(HANDBOOK)} handbook entries loaded")

## The brief

Four architectures. The same five help desk requests. One interface, so they are swappable and the
only thing that varies is how the arm reasons.

The comparison is the easy part. The part teams get wrong is deciding what "better" means
*after* they have seen the table, which is how the arm that reads best on average gets shipped
with a hole in it.

So the bar comes first, in Section 1, before any arm exists. Two ideas have to be in it:

* **a minimum pass rate** &mdash; how many of the five it has to get right, and
* **a disqualifying case** &mdash; something no average can buy back. `EHD-7004` asks for
  `admin-rights`, which is in `ROUTE_OUT`. The help desk never grants that. An arm that resolves it
  itself is rejected however well it scores elsewhere.

## Section 1 &mdash; The acceptance bar, written first

Two decisions, and you make both of them blind. That is the point: a bar you write after seeing
the results is not a bar, it is a description of the winner.

In [ ]:
def acceptance_bar() -> dict:
    """The rule every arm is judged by. Written before any arm has run."""
    # Five requests. A 60% bar accepts an arm that is wrong about two employees in five.
    lenient    = {"min_pass_rate": 0.60, "reject_if_resolves_route_out": True}
    defensible = {"min_pass_rate": 0.80, "reject_if_resolves_route_out": True}
    return defensible   # one wrong answer in five is the most we would defend


def judge(result: dict, bar: dict) -> str:
    """result: {"arm", "pass_rate", "resolved_route_out"} -> "accepted" or "rejected"."""
    clears_bar   = result["pass_rate"] >= bar["min_pass_rate"]
    disqualified = result["resolved_route_out"] and bar["reject_if_resolves_route_out"]

    # Two readings of an arm that granted admin rights and still scored top of the table:
    average_wins = "accepted" if clears_bar else "rejected"
    rule_wins    = "rejected" if disqualified else ("accepted" if clears_bar else "rejected")
    return rule_wins    # the disqualifying case is not an input to an average

In [ ]:
# --- Self-check: Section 1   (the bar, over hand-written results -- no arms, no model)
def verdict(rate: float, resolved_route_out: bool) -> str:
    return judge({"arm": "x", "pass_rate": rate, "resolved_route_out": resolved_route_out}, acceptance_bar())

check("3 of 5 right is not good enough for an employee who is one of the other two",
      lambda: verdict(0.60, False) == "rejected",
      "a 60% bar is the one you write when you already know the scores")
check("4 of 5 right, nothing wrongly resolved: accepted",
      lambda: verdict(0.80, False) == "accepted")
check("5 of 5 right but it granted admin rights itself: rejected anyway",
      lambda: verdict(1.00, True) == "rejected",
      "no pass-rate average buys back a request the help desk must never handle")
score()

## Section 2 &mdash; Four arms behind one interface

Each arm is the same LangChain shape &mdash; a `ChatPromptTemplate`, the model, a parser &mdash; and
differs only in the system line that tells it how to work. That is what makes the comparison fair:
same case template, same model, same output schema, one variable.

The output schema is a Pydantic model, so every arm has to answer in the same terms whatever it did
to get there. `Field(description=...)` is not a comment &mdash; it is what the model actually reads.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class Handling(BaseModel):
    """What every arm must produce, whatever reasoning style it used to get there."""
    action: Literal["resolve", "route_out"] = Field(
        description="resolve if the help desk handles it; route_out if another team must")
    sla_hours: int = Field(description="hours allowed, from SLA_HOURS unless a handbook rule is stricter")
    why: str = Field(description="one sentence naming the handbook rule you applied")


CASE = ("Request {rid}: {text}\n"
        "category={category} urgency={urgency} wants={wants}\n"
        "Handbook rule for this category: {rule}\n"
        "SLA hours by urgency: {sla}\n"
        "The help desk must NEVER do these itself: {route_out}")

STYLES = {
    "direct":  "Answer immediately. Do not show any working.",
    "cot":     "Work in order in your reply: category, then the handbook rule, then whether that "
               "rule overrides the urgency the employee set, then the action.",
    "react":   "Work in Thought / Action / Observation steps, then a final answer. Actions: "
               "handbook(category), sla(urgency).",
    "reflect": "Draft an answer, critique your own draft against the handbook rule, then correct it.",
}


def arm_prompt(style: str) -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages([
        ("system", "You triage employee help desk requests. " + STYLES[style]),
        ("human", CASE),
    ])

def build_arm(style: str, llm) -> dict:
    """One arm, two chains over the same prompt: the prose it wrote, the verdict we score."""
    prompt = arm_prompt(style)
    return {"prose":   prompt | llm | StrOutputParser(),
            "verdict": prompt | llm.with_structured_output(Handling),
            # no parser on this leg, so the AIMessage -- and its usage_metadata -- survives
            "raw":     prompt | llm}

def case_inputs() -> list:
    return [{"rid": rid, "text": r["text"], "category": r["category"], "urgency": r["urgency"],
             "wants": r["wants"], "rule": HANDBOOK[r["category"]],
             "sla": SLA_HOURS, "route_out": sorted(ROUTE_OUT)}
            for rid, r in sorted(REQUESTS.items())]

Now the selection rule. Four arms have run, you have a pass rate for each and whether each one
resolved `EHD-7004`. The table is in front of you &mdash; and this is exactly the moment the bar you
wrote in Section 1 has to be allowed to win.

In [ ]:
# What a correct handling looks like: (action, the slowest clock we would accept)
EXPECTED = {
    "EHD-7001": ("resolve",    4),
    "EHD-7002": ("resolve",    4),    # the safety rule, not the urgency box, sets this one
    "EHD-7003": ("resolve",   72),    # 180 USD needs approval first -- but it is still ours
    "EHD-7004": ("route_out", 24),    # the disqualifying case
    "EHD-7005": ("resolve",   72),
}

def graded(rid: str, h) -> bool:
    action, max_hours = EXPECTED[rid]
    return h is not None and h.action == action and h.sla_hours <= max_hours


def pick_winner(results: list, bar: dict):
    """results: {"arm", "pass_rate", "resolved_route_out", "tokens"}. Returns one, or None."""
    accepted = [r for r in results if judge(r, bar) == "accepted"]

    top_scorer       = sorted(results, key=lambda r: -r["pass_rate"])[0]
    best_accepted    = sorted(accepted, key=lambda r: -r["pass_rate"])[0] if accepted else None
    cheapest_accepted = sorted(accepted, key=lambda r: r["tokens"])[0] if accepted else None

    # Above the bar, extra quality is something you are paying for and not using. Ranking only
    # ever runs over arms that already cleared it, so cost is what is left to decide.
    return cheapest_accepted

In [ ]:
# --- Self-check: Section 2   (prompt objects, and the rule over results we typed by hand)
FIELDS = {"rid", "text", "category", "urgency", "wants", "rule", "sla", "route_out"}
RECORDED = [
    {"arm": "direct",  "pass_rate": 0.80, "resolved_route_out": False, "tokens":  2100},
    {"arm": "cot",     "pass_rate": 1.00, "resolved_route_out": False, "tokens":  5400},
    {"arm": "react",   "pass_rate": 0.80, "resolved_route_out": True,  "tokens":  6800},
    {"arm": "reflect", "pass_rate": 0.80, "resolved_route_out": False, "tokens": 11900},
]

check("all four arms are ChatPromptTemplates over one identical case",
      lambda: all(set(arm_prompt(s).input_variables) == FIELDS for s in STYLES) and len(STYLES) == 4,
      "if the arms saw different facts you would be comparing prompts, not architectures")
check("the schema tells the model what route_out means, in words it reads",
      lambda: "route_out" in Handling.model_fields["action"].description and "route_out" in str(Handling.model_json_schema()))
check("of the three arms that clear the bar, the cheapest is what ships",
      lambda: pick_winner(RECORDED, acceptance_bar())["arm"] == "direct",
      "cot scores higher and costs 2.5x -- above the bar you are paying for quality you said "
      "you did not need")
check("if the cheapest arm is the disqualified one, it is still not shipped",
      lambda: pick_winner([{**r, "tokens": 100 if r["arm"] == "react" else r["tokens"]}
                           for r in RECORDED], acceptance_bar())["arm"] != "react")
check("react is refused even when it alone tops the table",
      lambda: pick_winner([{**r, "pass_rate": 1.00 if r["arm"] == "react" else 0.80}
                           for r in RECORDED], acceptance_bar())["arm"] != "react",
      "sort first and you ship the disqualified arm -- filter first, then rank")
score()

## Run it for real &mdash; the bake-off

Four arms, five requests each, run with `.batch()` so the five go out together. One prose sample
first, so you can see what the arm actually wrote before you read it as a number.

In [ ]:
def bake_off():
    llm, inputs, bar = get_llm(), case_inputs(), acceptance_bar()
    print(f"bar: pass rate >= {bar['min_pass_rate']:.0%}; resolving {sorted(ROUTE_OUT)} disqualifies\n")
    print("--- chain-of-thought, in its own words, on EHD-7004 ---")
    print(build_arm("cot", llm)["prose"].invoke(inputs[3]).strip()[:420], "\n")

    results = []
    for style in STYLES:
        arm = build_arm(style, llm)
        out = arm["verdict"].batch(inputs)
        passed = sum(1 for i, h in zip(inputs, out) if graded(i["rid"], h))
        wrong  = any(h is not None and h.action == "resolve"
                     for i, h in zip(inputs, out) if i["rid"] == "EHD-7004")
        # what the arm cost: the prose leg carries the usage metadata the verdict leg hides
        msgs = arm["raw"].batch(inputs)
        toks = sum((getattr(m, "usage_metadata", None) or {}).get("total_tokens", 0) for m in msgs)
        results.append({"arm": style, "pass_rate": passed / len(inputs),
                        "resolved_route_out": wrong, "tokens": toks})

    print(f"{'arm':10} {'pass':>5} {'tokens':>8}  {'EHD-7004 resolved':>18}  verdict")
    for r in results:
        print(f"{r['arm']:10} {r['pass_rate']:>5.0%} {r['tokens']:>8}  "
              f"{str(r['resolved_route_out']):>18}  {judge(r, bar)}")

    win = pick_winner(results, bar)
    print("\nship:", win["arm"] if win else "nothing -- no arm cleared the bar")
    print("      (of the arms that cleared the bar, the cheapest one)")

if llm_ready():
    guard(bake_off)

### Read it

The shape we recorded on this sandbox: **direct 80%, chain-of-thought 100%, ReAct 80%, reflection
80%** &mdash; and ReAct **rejected**, because it resolved `EHD-7004` itself instead of routing the
admin-rights request out. Your run will not match it exactly. The same arm can score 80% on one
pass and 60% on the next with nothing changed; a small model is not a fixed function, and four
runs of five cases is far too little data to call one architecture better than another by two
percentage points.

**What does not move is the rule.** ReAct got rejected on a case, not on an average, and a case
verdict is stable in a way a mean of five is not. That asymmetry is the whole lab: put your
confidence in the disqualifying cases you can name, not in the third decimal place of a pass rate.
Notice also that `pick_winner` filters before it ranks. Rank first and the highest number wins,
which is precisely how a disqualified arm gets shipped.

**And look at the tokens column before you crown chain-of-thought.** It scores highest and costs
several times the direct arm. The bar already said what quality you need; above it, extra quality
is something you are paying for and not using &mdash; so `pick_winner` ships the *cheapest arm that
clears*, which on this set is usually the direct one. Chain-of-thought earns its cost only where it
changes the verdict: two of these five cases need two facts combined (`EHD-7002`'s safety rule
against the urgency box, `EHD-7003`'s 180 USD against the approval threshold). On a set of one-fact
requests it would buy you nothing and bill you for it.

That is also why the bar is written first. Decide the quality you need, then buy it as cheaply as
you can &mdash; rather than admiring the biggest number and calling it a decision.

**What you take from Module 2.** Reasoning style is a design choice with a measurable cost, not a
personality. Chain-of-thought makes the intermediate facts explicit; ReAct interleaves acting with
thinking and lives or dies on the argument contract, not the text format; reflection buys a second
look at your own draft. None of them is the default. You pick with an eval set, an acceptance bar
written before the results, and at least one case that no average is allowed to override. Module 3
turns the reasoning you chose here into a graph you can pause, inspect and resume.

In [ ]:
score()

## Your turn

1. Move `min_pass_rate` to `1.0` and re-run the selection. Most likely nothing ships. Decide which
   you would actually do &mdash; lower the bar, or keep the human in the loop for the cases no arm
   gets right &mdash; and say what your answer implies about who carries the risk.
2. Add a sixth request that also `wants` something in `ROUTE_OUT`, and re-run. If an arm routes one
   out and resolves the other, is that a pass rate of 50% on those two, or still a disqualification?
   Your answer is a change to `judge`, so make it.